# 16 · CodeMode：代码编排与工具执行是两层并发

一次 cell 可以发起多个工具调用，但 cell 本身不等于一项工具动作。工具权限、并发准入、结果和副作用必须各自记账。
本章的 Python Cell 是生命周期教学模型，不执行 JavaScript，也没有 V8 或隔离 worker；它通过真实文件和真实子进程，
展示 ready 闸门、yield 观察、按实际目标授权、部分失败，以及取消后已经发生的文件写入。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [S16-03 · CodeModeExecuteHandler::execute](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/code_mode/execute_handler.rs#L96-L130)：cell 创建后先登记 executed calls 和 trace，再标记 ready；broker 分派会等待 ready 或取消。
- [S16-10 · ToolCallRuntime::handle_tool_call_with_source](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/core/src/tools/parallel.rs#L161-L195)：并行资格来自 router.tool_supports_parallel；一个 ToolCallRuntime 的 read gate 可并行，write gate 排他；等待准入仍可取消。
- [S16-13 · SessionRuntime::start_cell / RuntimeCellHost::commit_completion](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/code-mode-runtime/src/session_runtime/mod.rs#L163-L194)：store/load 通过 JSON 值跨 cell 传递；cell 启动拷贝 session map 快照，完成提交时仅合并本 cell 写集；Result 可携带 error_text，脚本错误不自动回滚该写集。


## 实验 1 · 先登记 cell，再放行两个读工具


ready 前可以接受调用身份，不能让 handler 先于登记执行。ready 后两个读调用同时在途，说明调度准入允许它们并行；
这不意味着任意写工具也安全并行。教学版使用容量 3 的 semaphore，固定 Codex 还区分工具的并行资格与读写 gate。


In [2]:
cm = load_example('16', 'scheduler_probe')
import asyncio, json
run_dir = fresh_run_dir('ch16')
(run_dir/'tasks.json').write_text(json.dumps({'tasks':['add','list']}))
(run_dir/'README.md').write_text('task-list: add and list tasks\n')
runtime, session = cm.Runtime(run_dir), {}
cell = cm.Cell('cell-reads', runtime, session)
a = cell.start_call('read', 'tasks.json')
b = cell.start_call('read', 'README.md')
await asyncio.sleep(0)
assert len(runtime.ledger) == 2
assert all(row['handler_count'] == 0 for row in runtime.ledger)
cell.ready.set()
await asyncio.wait_for(runtime.both_reads.wait(), 2)
assert not a.done() and not b.done()
print({'inflight_reads': runtime.reads_started, 'handlers_started': [r['handler_count'] for r in runtime.ledger]})


{'inflight_reads': 2, 'handlers_started': [1, 1]}


## 实验 2 · yield 只交还观察，随后接收增量结果


一次 observe 只取上次游标之后的输出，不能把 yielded 当 completed。
现在提交真实 Python 检查进程，释放文件读，再汇合结果。把 tasks 改成三个元素会使固定检查失败，运行下一格可单独观察该反例。


In [3]:
cell.output.append('A: reads started')
first = cell.observe()
assert first['kind'] == 'yielded' and not a.done()
assert cell.observe()['items'] == []
check_process = cell.start_call('check', 'tasks.json')
runtime.release_reads.set()
results = await asyncio.wait_for(asyncio.gather(a, b, check_process), 3)
assert json.loads(results[0])['tasks'] == ['add','list']
assert 'add and list' in results[1]
assert results[2]['exit_code'] == 0 and results[2]['stdout'] == 'CHECK_OK\n'
cell.output.append('B: process check finished')
second = cell.observe()
assert second['items'] == ['B: process check finished']
assert (await cell.close())['kind'] == 'completed'
print({'first': first, 'second': second, 'real_process': results[2]})


{'first': {'cell_id': 'cell-reads', 'kind': 'yielded', 'items': ['A: reads started']}, 'second': {'cell_id': 'cell-reads', 'kind': 'yielded', 'items': ['B: process check finished']}, 'real_process': {'exit_code': 0, 'stdout': 'CHECK_OK\n', 'stderr': ''}}


## 实验 3 · 失败结果属于各自调用，不吞掉成功兄弟


一个子调用失败时，保留另一个的有效观察。否则只返回“整个 cell 报错”，模型就可能重做已经成功的操作。
同时用错误输入运行真实检查进程；日志中应保留 failed，而不是依据 cell 自然结束报告成功。


In [4]:
partial = cm.Cell('partial', runtime, session)
partial.ready.set()
values = await asyncio.gather(partial.start_call('read','missing.txt'),
                              partial.start_call('read','README.md'), return_exceptions=True)
assert isinstance(values[0], FileNotFoundError) and isinstance(values[1], str)
(run_dir/'bad-tasks.json').write_text(json.dumps({'tasks':['one']}))
with expect.assertRaises(RuntimeError) as failed:
    await partial.start_call('check', 'bad-tasks.json')
process_failure = failed.exception.args[0]
assert process_failure['exit_code'] != 0 and 'AssertionError' in process_failure['stderr']
assert runtime.ledger[-1]['outcome'] == 'failed'
await partial.close()
print({'sibling_preserved': values[1].strip(), 'failure_exit_code': process_failure['exit_code']})


{'sibling_preserved': 'task-list: add and list tasks', 'failure_exit_code': 1}


## 实验 4 · Hook 改写目标后重新授权


rewrite.txt 在教学 Hook 中被改为 protected.txt。若仍用改写前路径批准，nested 就会变成绕过 direct 权限的后门。
两种入口复用 Runtime.invoke，检查 handler_count=0 可证明拒绝发生在实际 handler 前。


In [5]:
denied = cm.Cell('denials', runtime, session)
denied.ready.set()
for path in ('protected.txt','rewrite.txt'):
    for origin in ('direct','nested'):
        with expect.assertRaisesRegex(PermissionError, 'protected_target'):
            if origin == 'direct':
                await runtime.invoke('read', path, origin='direct')
            else:
                await denied.start_call('read', path)
rows = runtime.ledger[-4:]
assert all(r['actual'] == 'protected.txt' and r['handler_count'] == 0 for r in rows)
assert 'task-list' in await denied.start_call('read','README.md')
await denied.close()
print([{'origin':r['origin'], 'requested':r['requested'], 'actual':r['actual']} for r in rows])


[{'origin': 'direct', 'requested': 'protected.txt', 'actual': 'protected.txt'}, {'origin': 'nested', 'requested': 'protected.txt', 'actual': 'protected.txt'}, {'origin': 'direct', 'requested': 'rewrite.txt', 'actual': 'protected.txt'}, {'origin': 'nested', 'requested': 'rewrite.txt', 'actual': 'protected.txt'}]


## 实验 5 · store 是值快照，取消不能回滚外部写入


store/load 复制 JSON 值，不共享活对象；正常完成提交写集，terminate 不提交它。
但文件写入由工具拥有：在写已提交、回执尚未返回的时刻取消，工具状态必须是 unknown，文件仍存在。
恢复时先观察副作用，不自动重放那次写入。


In [6]:
writer = cm.Cell('store-writer', runtime, session)
original = {'items':[1]}
writer.store('memo', original)
original['items'].append(2)
assert writer.load('memo') == {'items':[1]}
await writer.close()
next_cell = cm.Cell('store-reader', runtime, session)
copy = next_cell.load('memo')
copy['items'].append(3)
assert next_cell.load('memo') == {'items':[1]}
await next_cell.close()
ambiguous = cm.Cell('cancel-after-write', runtime, session)
ambiguous.ready.set()
ambiguous.store('uncommitted','discard me')
ambiguous.start_call('write_then_block','effect.txt')
await asyncio.wait_for(runtime.write_committed.wait(), 2)
await ambiguous.close(terminate=True)
assert runtime.ledger[-1]['outcome'] == 'unknown'
assert runtime.ledger[-1]['handler_count'] == 1
assert (run_dir/'effect.txt').read_text() == 'committed once\n'
assert 'uncommitted' not in session
with expect.assertRaisesRegex(RuntimeError, 'cell_closed'):
    ambiguous.start_call('read','README.md')
print({'tool_outcome': runtime.ledger[-1]['outcome'], 'effect_exists': True, 'session': session})


{'tool_outcome': 'unknown', 'effect_exists': True, 'session': {'memo': {'items': [1]}}}


## 你现在可以解释什么

能分别说明 cell 状态、每项工具结果、session 写集和外部副作用，且不会把 Promise 并发当成执行器允许并发。本章没有 JavaScript 执行、gRPC、OS sandbox 或真实模型。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
